# 🚖 下車地址推薦系統

**架構：兩階段推薦**
- Stage 1 (Candidate Generation)：從 `address_v2_suggestion` 撈出用戶歷史下車地址候選集
- Stage 2 (Ranking)：用 **LightGBM LambdaRank** 對候選集排序，輸出 Top-K 推薦

**特徵（12 個）：** 原始頻率 × 6 + log1p 轉換版 × 6

| 特徵 | 說明 |
|---|---|
| `user_end_freq` | 用戶歷史去過該地址幾次（最強信號）|
| `global_end_freq` | 全局熱門度（cold-start 用）|
| `hour_end_freq` | 同時段去該地址的條件頻率 |
| `holiday_end_freq` | 同假日/平日狀態的條件頻率 |
| `dow_end_freq` | 同星期幾的條件頻率 |
| `start_end_freq` | 從同一上車區域出發去該地址的頻率 |

**評估指標：** Recall@K、MRR@K、NDCG@K（K = 1, 3, 5）

---
### 使用說明
1. 把 `address_v2_training_data.parquet` 和 `address_v2_suggestion.parquet` 上傳到 Google Drive
2. 在 **⚙️ 設定** 那格填入正確的檔案路徑
3. 依序執行所有 Cell（`執行階段 → 全部執行`）

## 0｜安裝套件

In [1]:
# lightgbm --upgrade 確保版本支援 GPU
!pip install lightgbm --upgrade tqdm pyarrow -q

# 把同學的工具檔案上傳到 Colab 工作目錄（若已上傳可跳過）
# 需要: data_loader.py / evaluate.py / read_parquet.py
# 放在 /content/ 底下即可，或直接上傳到左側檔案面板


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.6 MB/s eta 0:00:00


## 1｜掛載 Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive 已掛載')

Mounted at /content/drive
✓ Google Drive 已掛載


## ⚙️ 設定（請修改這格）

In [3]:
from pathlib import Path
import sys

# ── Google Drive 路徑 ────────────────────────────────────────────────
DRIVE_ROOT  = Path('/content/drive/MyDrive')
DATA_FOLDER = DRIVE_ROOT / 'LineGO_data'   # ← 改成你放資料的資料夾
OUTPUT_DIR  = DRIVE_ROOT / 'LineGO_checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 把同學的工具檔加進 sys.path（假設放在 /content/）───────────────────
# 若放在其他路徑，修改下面這行
UTILS_DIR = Path('/content')
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

# ── 讓 read_parquet.py 找得到資料（它用 Path(__file__).parent / '1.下車地址推薦'）
# 把兩個 parquet 放在 /content/1.下車地址推薦/ 或修改 read_parquet.py 的 BASE_DIR
import os
DATA_LINK = Path('/content/1.下車地址推薦')
if not DATA_LINK.exists():
    DATA_LINK.mkdir(parents=True, exist_ok=True)
    # 從 Drive 建 symlink（不複製，省空間）
    for fname in ['address_v2_training_data.parquet', 'address_v2_suggestion.parquet']:
        src = DATA_FOLDER / fname
        dst = DATA_LINK / fname
        if src.exists() and not dst.exists():
            os.symlink(src, dst)

# ── 模型超參數 ────────────────────────────────────────────────────────
TOP_K_LIST = [1, 3, 5]
NEG_RATIO  = 4      # 每個正樣本配幾個負樣本
N_ROUNDS   = 500    # LightGBM 最大迭代數
EARLY_STOP = 30     # val NDCG 連續幾輪不進步就停止
SEED       = 42

FEAT_COLS = ['user_end_freq', 'global_end_freq', 'hour_end_freq',
             'holiday_end_freq', 'dow_end_freq', 'start_end_freq']
ALL_FEAT  = FEAT_COLS + [f'log_{c}' for c in FEAT_COLS]

print('✓ 設定完成')


✓ 設定完成


## 2｜Import

In [4]:
import gc, json, pickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from tqdm.notebook import tqdm

# 同學的工具模組
from data_loader import load_split      # 時序切分，全量用戶
from evaluate import evaluate as eval_topk, EvalResult  # row-level 評估

print('✓ 所有套件載入完成')


✓ 所有套件載入完成


## 2.5｜GPU 偵測

In [5]:
import subprocess

def detect_gpu():
    """偵測是否有可用的 GPU，並設定 LightGBM 訓練裝置。"""
    global USE_GPU, LGB_DEVICE_PARAMS

    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
             '--format=csv,noheader'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0 and result.stdout.strip():
            gpu_info = result.stdout.strip().split('\n')
            print('🟢 GPU 可用！')
            for i, info in enumerate(gpu_info):
                name, mem, driver = [x.strip() for x in info.split(',')]
                print(f'   GPU {i}: {name} | 顯存: {mem} | Driver: {driver}')
            USE_GPU = True
            LGB_DEVICE_PARAMS = {
                'device':          'gpu',
                'gpu_platform_id': 0,
                'gpu_device_id':   0,
            }
            print('   ✓ LightGBM 將使用 GPU 訓練')
        else:
            raise RuntimeError('nvidia-smi 無回應')
    except Exception as e:
        print(f'🔴 未偵測到 GPU（{e}）')
        print('   → 使用 CPU 訓練（可至 執行階段 → 變更執行階段類型 → T4 GPU 切換）')
        USE_GPU = False
        LGB_DEVICE_PARAMS = {}  # 空 dict = CPU 模式，不改任何參數

    # 額外驗證：讓 LightGBM 實際試跑一個最小 GPU dataset
    if USE_GPU:
        try:
            import lightgbm as lgb, numpy as np
            _X = np.random.rand(100, 4).astype(np.float32)
            _y = np.random.randint(0, 2, 100).astype(np.float32)
            _ds = lgb.Dataset(_X, label=_y)
            lgb.train(
                {'objective': 'binary', 'verbosity': -1, **LGB_DEVICE_PARAMS},
                _ds, num_boost_round=3
            )
            print('   ✓ LightGBM GPU 驗證通過')
        except Exception as e:
            print(f'   ⚠️  LightGBM GPU 驗證失敗：{e}')
            print('   → 自動退回 CPU 模式')
            USE_GPU = False
            LGB_DEVICE_PARAMS = {}

USE_GPU = False
LGB_DEVICE_PARAMS = {}
detect_gpu()

🟢 GPU 可用！
   GPU 0: Tesla T4 | 顯存: 15360 MiB | Driver: 580.82.07
   ✓ LightGBM 將使用 GPU 訓練
   ✓ LightGBM GPU 驗證通過


## 3｜讀取資料

In [6]:
def load_data():
    """
    使用 data_loader.load_split() 做時序切分（全量用戶，不過濾 MIN_TRIPS）。
    切分邏輯: 全量行程依 created_at 排序 → train 75% / val 10% / test 15%
    """
    print('[1/5] 讀取資料（使用 data_loader）...')
    with tqdm(total=3, desc='  載入 splits', unit='split') as pbar:
        pbar.set_postfix_str('train')
        df_train_raw = load_split('train')
        pbar.update(1)

        pbar.set_postfix_str('val')
        df_val_raw = load_split('val')
        pbar.update(1)

        pbar.set_postfix_str('test')
        df_test_raw = load_split('test')
        pbar.update(1)

    # 讀 suggestion 表（地址對照用）
    table   = pq.read_table(DATA_LINK / 'address_v2_suggestion.parquet')
    df_sugg = pd.DataFrame({c: table.column(c).to_pylist() for c in table.column_names})
    del table; gc.collect()

    def _info(name, df):
        t0 = df['created_at'].min().strftime('%Y/%m/%d')
        t1 = df['created_at'].max().strftime('%Y/%m/%d')
        print(f'  {name:<8} {len(df):>7,} 筆  '
              f'{df["uid_hash"].nunique():>6,} 用戶  {t0} ~ {t1}')

    print()
    _info('train',  df_train_raw)
    _info('val',    df_val_raw)
    _info('test',   df_test_raw)
    print(f'  suggestion  {len(df_sugg):>7,} 筆')
    return df_train_raw, df_val_raw, df_test_raw, df_sugg

df_train_raw, df_val_raw, df_test_raw, df_sugg = load_data()


[1/5] 讀取資料（使用 data_loader）...


  載入 splits:   0%|          | 0/3 [00:00<?, ?split/s]


  train    750,000 筆  273,683 用戶  2026/01/01 ~ 2026/04/14
  val      100,000 筆  65,461 用戶  2026/04/14 ~ 2026/04/28
  test     150,000 筆  89,435 用戶  2026/04/28 ~ 2026/05/17
  suggestion  2,493,639 筆


## 4｜預計算頻率查找表

In [7]:
def build_lookup_tables(df_train_raw: pd.DataFrame) -> dict:
    """
    只用 train 行程計算頻率查找表（全量用戶版）。
    不在 fit 時過濾用戶，讓所有用戶的訓練行程都參與統計。
    """
    lookup_defs = [
        ('user_end',    ['uid_hash',     'end_latlng']),
        ('global_end',  ['end_latlng']),
        ('hour_end',    ['hour_type',    'end_latlng']),
        ('holiday_end', ['is_holiday',   'end_latlng']),
        ('dow_end',     ['dayofweek',    'end_latlng']),
        ('start_end',   ['start_latlng', 'end_latlng']),
    ]
    lookups = {}
    with tqdm(lookup_defs, desc='[2/5] 建立頻率查找表 (僅用 train)', unit='table') as pbar:
        for name, keys in pbar:
            pbar.set_postfix_str(name)
            lookups[name] = df_train_raw.groupby(keys).size().to_dict()

    print(f'  ✓ lookup sizes: {", ".join(f"{k}={len(v):,}" for k,v in lookups.items())}')
    return lookups

lookups = build_lookup_tables(df_train_raw)


[2/5] 建立頻率查找表 (僅用 train):   0%|          | 0/6 [00:00<?, ?table/s]

  ✓ lookup sizes: user_end=533,619, global_end=66,988, hour_end=182,628, holiday_end=99,676, dow_end=202,226, start_end=407,139


## 5｜建立訓練樣本

In [8]:
def _get_feat(lookups, uid, end, hour, holiday, dow, start):
    f = [
        lookups['user_end'].get((uid, end), 0),
        lookups['global_end'].get(end, 0),
        lookups['hour_end'].get((hour, end), 0),
        lookups['holiday_end'].get((holiday, end), 0),
        lookups['dow_end'].get((dow, end), 0),
        lookups['start_end'].get((start, end), 0),
    ]
    return f + [np.log1p(x) for x in f]


def _df_to_samples(df_raw, df_sugg, lookups, split_name):
    """
    將某個 split 的行程轉成訓練樣本列表。
    候選集來自 suggestion 表（該用戶曾去過的地址）。
    沒有候選集的用戶（新用戶）這裡跳過，predict 時用 fallback 處理。
    """
    sugg_dict = (df_sugg.groupby('uid_hash')['end_latlng']
                        .apply(lambda g: g.drop_duplicates().tolist())
                        .to_dict())
    rows = []
    groups = list(df_raw.groupby('uid_hash', sort=False))
    with tqdm(groups, desc=f'  {split_name}', unit='user', leave=False) as pbar:
        for uid, grp in pbar:
            cands = sugg_dict.get(uid, [])
            if not cands: continue
            cand_set = set(cands)
            for _, trip in grp.iterrows():
                true_end = trip['end_latlng']
                if true_end not in cand_set: continue
                neg_cands = [c for c in cands if c != true_end]
                n_neg = min(len(neg_cands), NEG_RATIO)
                sel   = (np.random.choice(len(neg_cands), n_neg, replace=False)
                         if n_neg > 0 else [])
                h, hol, dow, start = (trip['hour_type'], trip['is_holiday'],
                                      trip['dayofweek'], trip['start_latlng'])
                rows.append((uid, 1, _get_feat(lookups, uid, true_end, h, hol, dow, start)))
                for i in sel:
                    rows.append((uid, 0, _get_feat(lookups, uid, neg_cands[i], h, hol, dow, start)))
    return rows


def to_df(rows):
    feats = np.array([r[2] for r in rows], dtype=np.float32)
    df = pd.DataFrame(feats, columns=ALL_FEAT)
    df.insert(0, 'label',    [r[1] for r in rows])
    df.insert(0, 'uid_hash', [r[0] for r in rows])
    return df


def build_samples(df_train_raw, df_val_raw, df_test_raw, df_sugg, lookups):
    print('[3/5] 建立訓練樣本...')
    np.random.seed(SEED)
    rows_tr   = _df_to_samples(df_train_raw, df_sugg, lookups, 'train')
    rows_val  = _df_to_samples(df_val_raw,   df_sugg, lookups, 'val')
    rows_test = _df_to_samples(df_test_raw,  df_sugg, lookups, 'test')

    df_train = to_df(rows_tr)
    df_val   = to_df(rows_val)
    df_test  = to_df(rows_test)

    for name, df in [('train', df_train), ('val', df_val), ('test', df_test)]:
        print(f'  ✓ {name:<6} {len(df):>7,} 樣本  pos_rate={df["label"].mean():.3f}')

    df_train.to_parquet(OUTPUT_DIR / 'samples_train.parquet', index=False)
    df_val.to_parquet(OUTPUT_DIR   / 'samples_val.parquet',   index=False)
    df_test.to_parquet(OUTPUT_DIR  / 'samples_test.parquet',  index=False)
    print('  ✓ 樣本已存至 Drive')
    return df_train, df_val, df_test


df_train, df_val, df_test = build_samples(
    df_train_raw, df_val_raw, df_test_raw, df_sugg, lookups
)


[3/5] 建立訓練樣本...


  train:   0%|          | 0/273683 [00:00<?, ?user/s]

  val:   0%|          | 0/65461 [00:00<?, ?user/s]

  test:   0%|          | 0/89435 [00:00<?, ?user/s]

  ✓ train  3,227,255 樣本  pos_rate=0.232
  ✓ val    428,468 樣本  pos_rate=0.233
  ✓ test   639,202 樣本  pos_rate=0.235
  ✓ 樣本已存至 Drive


## 6｜訓練 LightGBM LambdaRank

In [9]:
def _tqdm_lgb_callback(pbar):
    """把 LightGBM iteration 進度接到 tqdm 進度條，並即時顯示 val NDCG@1。"""
    prev = [0]
    def callback(env):
        n = env.iteration + 1
        pbar.update(n - prev[0])
        prev[0] = n
        # evaluation_result_list 順序：[train_ndcg@1, val_ndcg@1, ...]
        # 取最後一個 dataset（val）的 NDCG@1
        if env.evaluation_result_list:
            val_metric = env.evaluation_result_list[-3]  # val NDCG@1
            pbar.set_postfix_str(f'val NDCG@1={val_metric[2]:.4f}')
    callback.order = 10
    return callback


def train_model(df_train: pd.DataFrame, df_val: pd.DataFrame) -> lgb.Booster:
    print('[4/5] 訓練 LightGBM LambdaRank...')

    X_tr  = df_train[ALL_FEAT].values
    y_tr  = df_train['label'].values
    g_tr  = df_train.groupby('uid_hash', sort=False).size().values

    X_val = df_val[ALL_FEAT].values
    y_val = df_val['label'].values
    g_val = df_val.groupby('uid_hash', sort=False).size().values

    ds_train = lgb.Dataset(X_tr,  label=y_tr,  group=g_tr,  feature_name=ALL_FEAT)
    ds_val   = lgb.Dataset(X_val, label=y_val, group=g_val, feature_name=ALL_FEAT,
                           reference=ds_train)  # reference 避免重複 bin 計算

    params = {
        'objective':        'lambdarank',
        'metric':           'ndcg',
        'ndcg_eval_at':     [1, 3, 5],
        'learning_rate':    0.05,
        'num_leaves':       63,
        'min_data_in_leaf': 10,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq':     5,
        'verbosity':        -1,
        'seed':             SEED,
        **LGB_DEVICE_PARAMS,
    }
    print(f'  裝置模式: {"GPU 🟢" if USE_GPU else "CPU 🔵"}')
    print(f'  最大迭代: {N_ROUNDS} rounds，early stopping: {EARLY_STOP} rounds')

    ckpt_path  = OUTPUT_DIR / 'lgbm_checkpoint.txt'
    meta_path  = OUTPUT_DIR / 'train_meta.json'
    init_model = str(ckpt_path) if ckpt_path.exists() else None
    start_iter = 0
    if init_model:
        if meta_path.exists():
            with open(meta_path) as f:
                start_iter = json.load(f).get('n_iter', 0)
        print(f'  發現 checkpoint，從第 {start_iter} 輪繼續...')

    with tqdm(total=start_iter + N_ROUNDS, initial=start_iter,
              desc='  迭代訓練', unit='round') as pbar:
        model = lgb.train(
            params,
            ds_train,
            num_boost_round=N_ROUNDS,
            valid_sets=[ds_train, ds_val],
            valid_names=['train',  'val'],
            callbacks=[
                lgb.log_evaluation(-1),
                lgb.early_stopping(EARLY_STOP, verbose=False),
                _tqdm_lgb_callback(pbar),
            ],
            init_model=init_model,
        )

    best = model.best_iteration
    print(f'\n  ✓ Early stopping 於第 {best} 輪停止')

    # 儲存 checkpoint 到 Drive
    model.save_model(str(ckpt_path))
    with open(OUTPUT_DIR / 'lgbm_ranker.pkl', 'wb') as f:
        pickle.dump(model, f)
    with open(meta_path, 'w') as f:
        json.dump({'n_iter': model.num_trees(), 'best_iteration': best,
                   'feature_names': ALL_FEAT}, f)

    print('  Feature importance (gain):')
    pairs = sorted(zip(ALL_FEAT, model.feature_importance('gain')), key=lambda x: -x[1])
    max_s = pairs[0][1]
    for name, score in pairs:
        bar = '█' * int(score / max_s * 25)
        print(f'    {name:<30} {bar}  {score:.0f}')
    return model


model = train_model(df_train, df_val)

[4/5] 訓練 LightGBM LambdaRank...
  裝置模式: GPU 🟢
  最大迭代: 500 rounds，early stopping: 30 rounds
  發現 checkpoint，從第 1 輪繼續...


  迭代訓練:   0%|          | 1/501 [00:00<?, ?round/s]


  ✓ Early stopping 於第 34 輪停止
  Feature importance (gain):
    log_user_end_freq              █████████████████████████  3128602
    user_end_freq                  ██████████  1289251
    start_end_freq                 ███  438371
    log_start_end_freq             ██  330548
    log_global_end_freq              124883
    global_end_freq                  111097
    hour_end_freq                    54219
    log_hour_end_freq                43960
    dow_end_freq                     31130
    log_dow_end_freq                 21450
    holiday_end_freq                 3485
    log_holiday_end_freq             2199


## 7｜評估模型

In [10]:
def predict_topk(model: lgb.Booster, query_df: pd.DataFrame,
                  df_sugg: pd.DataFrame, lookups: dict, k: int) -> list[list[str]]:
    """
    對 query_df 每一列 row 輸出 top-k 個 end_latlng，
    格式與同學的 baselines.predict_topk() 完全相同。

    候選集 = suggestion 表中該用戶的歷史地址。
    若用戶無候選集（新用戶）→ fallback 到全局最熱門地址。
    """
    sugg_dict = (df_sugg.groupby('uid_hash')[['end_latlng']]
                        .apply(lambda g: g['end_latlng'].drop_duplicates().tolist())
                        .to_dict())
    global_top = [x for x, _ in
                  sorted(lookups['global_end'].items(), key=lambda x: -x[1])[:50]]

    out: list[list[str]] = []
    cols = query_df[['uid_hash', 'start_latlng', 'hour_type',
                     'is_holiday', 'dayofweek']].values

    for uid, start, hour, hol, dow in tqdm(cols, desc='  predict', leave=False):
        cands = sugg_dict.get(uid, [])

        if not cands:
            # 新用戶 cold-start：global 熱門，排除上車點
            picks = [x for x in global_top if x != start][:k]
            out.append(picks)
            continue

        # 對所有候選地址計算特徵並打分
        feat_rows = [
            _get_feat(lookups, uid, c, hour, hol, dow, start)
            for c in cands
        ]
        scores = model.predict(np.array(feat_rows, dtype=np.float32))

        # 排序取 top-k，排除上車點
        ranked = sorted(zip(cands, scores), key=lambda x: -x[1])
        picks  = [c for c, _ in ranked if c != start][:k]

        # 不夠 k 個時用 global 補
        if len(picks) < k:
            seen = set(picks)
            for x in global_top:
                if x != start and x not in seen:
                    picks.append(x)
                    if len(picks) == k: break

        out.append(picks)
    return out


def run_evaluate(model, query_df, df_sugg, lookups,
                 split_name='test', k_list=TOP_K_LIST):
    """
    用 evaluate.py 的 evaluate() 做 row-level 評估。
    truths = query_df['end_latlng']（每筆行程真正的下車地址）。
    """
    print(f'[5/5] 評估模型（{split_name} split）...')
    max_k = max(k_list)

    with tqdm(total=2, desc='  評估', unit='step') as pbar:
        pbar.set_postfix_str('推論中')
        predictions = predict_topk(model, query_df, df_sugg, lookups, k=max_k)
        pbar.update(1)

        pbar.set_postfix_str('計算指標')
        truths = query_df['end_latlng'].tolist()
        pbar.update(1)

    print(f'\n  評估筆數: {len(truths):,} 筆（每筆為一個獨立 query）')
    print(f'  {"K":<5} {"Hit@K":>10} {"MRR":>10} {"NDCG@K":>10}')
    print(f'  {"-"*38}')

    out = {}
    for k in k_list:
        res = eval_topk(predictions, truths, k=k)
        print(f'  {k:<5} {res.hit_at_k:>10.4f} {res.mrr:>10.4f} {res.ndcg_at_k:>10.4f}')
        out[f'Hit@{k}']  = res.hit_at_k
        out[f'MRR@{k}']  = res.mrr
        out[f'NDCG@{k}'] = res.ndcg_at_k

    import json as _json
    with open(OUTPUT_DIR / f'eval_{split_name}.json', 'w') as f:
        _json.dump(out, f, indent=2)
    print(f'\n  ✓ 評估結果已存至 Drive (eval_{split_name}.json)')
    return predictions, out


# 先在 val 上評估，確認 early stopping 效果
val_preds, val_results = run_evaluate(
    model, df_val_raw, df_sugg, lookups, split_name='val'
)
# 最終 test 評估
test_preds, test_results = run_evaluate(
    model, df_test_raw, df_sugg, lookups, split_name='test'
)


[5/5] 評估模型（val split）...


  評估:   0%|          | 0/2 [00:00<?, ?step/s]

  predict:   0%|          | 0/100000 [00:00<?, ?it/s]


  評估筆數: 100,000 筆（每筆為一個獨立 query）
  K          Hit@K        MRR     NDCG@K
  --------------------------------------
  1         0.3617     0.3617     0.3617
  3         0.5979     0.4642     0.4984
  5         0.7228     0.4926     0.5498

  ✓ 評估結果已存至 Drive (eval_val.json)
[5/5] 評估模型（test split）...


  評估:   0%|          | 0/2 [00:00<?, ?step/s]

  predict:   0%|          | 0/150000 [00:00<?, ?it/s]


  評估筆數: 150,000 筆（每筆為一個獨立 query）
  K          Hit@K        MRR     NDCG@K
  --------------------------------------
  1         0.3495     0.3495     0.3495
  3         0.5917     0.4545     0.4896
  5         0.7185     0.4834     0.5418

  ✓ 評估結果已存至 Drive (eval_test.json)


## 8｜推薦示範

In [11]:
# ── 推薦示範 ────────────────────────────────────────────────────────
sample_row = df_test_raw.iloc[0]
uid = sample_row['uid_hash']
recs_raw = predict_topk(model, df_test_raw.iloc[[0]], df_sugg, lookups, k=5)

# end_latlng → end_address 對照
addr_map = df_sugg.drop_duplicates('end_latlng').set_index('end_latlng')['end_address'].to_dict()

print(f'用戶: {uid[:20]}...')
print(f'{"排名":<4} {"下車地址":<40} {"end_latlng"}')
print('-' * 70)
for i, latlng in enumerate(recs_raw[0], 1):
    addr = addr_map.get(latlng, latlng)
    print(f'#{i:<3} {addr:<40} {latlng}')

# ── per-segment 分析（依用戶歷史筆數） ───────────────────────────────
print('\n=== Per-segment 分析（用戶歷史筆數）===')
from evaluate import evaluate_by_segment, user_freq_bucket

truths_test = df_test_raw['end_latlng'].tolist()
seg = user_freq_bucket(df_train_raw, df_test_raw)
seg_df = evaluate_by_segment(test_preds, truths_test, seg, k=5)
print(seg_df.to_string(index=False))
print('\n說明：new(0)=訓練期間沒有行程的新用戶, 1-5=極少歷史, 6-20=中等, 21-100/100+=重度用戶')


  predict:   0%|          | 0/1 [00:00<?, ?it/s]

用戶: c930394ee75d85eb6d66...
排名   下車地址                                     end_latlng
----------------------------------------------------------------------
#1   臺北市大安區信義路四段58號                           25.033,121.545
#2   臺北市信義區基隆路二段149號                          25.027,121.556
#3   臺北市信義區基隆路一段163號                          25.042,121.565
#4   新北市三重區五華街200號                            25.087,121.488
#5   HERE by As...(臺北市大安區光復南路280巷43號1樓)       25.04,121.556

=== Per-segment 分析（用戶歷史筆數）===
segment     n  Hit@5    MRR  NDCG@5
    1-5 55804 0.6745 0.3767  0.4501
 new(0) 42768 0.9239 0.6392  0.7107
   6-20 37074 0.5831 0.4471  0.4808
 21-100 14298 0.6288 0.5293  0.5542
   100+    56 0.1786 0.1250  0.1385

說明：new(0)=訓練期間沒有行程的新用戶, 1-5=極少歷史, 6-20=中等, 21-100/100+=重度用戶


---
## 📥 只想重新評估（已有 checkpoint）
如果 runtime 重啟、模型已存在 Drive，執行以下這格就好，不用重新跑全部。

In [13]:
# ── 從 Drive 載入已訓練模型並重新評估（eval-only 模式）──────────────
with open(OUTPUT_DIR / 'lgbm_ranker.pkl', 'rb') as f:
    model_loaded = pickle.load(f)

# 重新載入資料（若 runtime 重啟）
df_train_raw = load_split('train')
df_val_raw   = load_split('val')
df_test_raw  = load_split('test')

with pq.read_table(DATA_LINK / 'address_v2_suggestion.parquet') as t:
    df_sugg = pd.DataFrame({c: t.column(c).to_pylist() for c in t.column_names})

lookups = build_lookup_tables(df_train_raw)

val_preds,  val_results  = run_evaluate(model_loaded, df_val_raw,  df_sugg, lookups, 'val')
test_preds, test_results = run_evaluate(model_loaded, df_test_raw, df_sugg, lookups, 'test')


TypeError: 'pyarrow.lib.Table' object does not support the context manager protocol